In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import joblib
import shap
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LinearRegression, Ridge, Lasso, RidgeCV, LassoCV
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, r2_score

os.chdir(r"C:\Users\camde\Desktop\baseball_analytics")
df = pd.read_csv(r"data\processed\batting_cleaned.csv")
print(df.columns.tolist())
print(df.head)


['playerID', 'yearID', 'G', 'AB', 'R', 'H', '2B', '3B', 'HR', 'RBI', 'SB', 'CS', 'BB', 'SO', 'IBB', 'HBP', 'SH', 'SF', 'GIDP', 'birthYear', 'nameLast', 'nameGiven', 'weight', 'height', 'lgID', 'salary', 'age', 'OBP', 'SLG', 'OPS', 'next_OPS', 'bats_B', 'bats_L', 'bats_R']
<bound method NDFrame.head of        playerID  yearID    G   AB    R    H  2B  3B  HR   RBI  ...  lgID  \
0     abbotku01    1994  101  345   41   86  17   3   9  33.0  ...    NL   
1     abbotku01    1995  120  420   60  107  18   7  17  60.0  ...    NL   
2     abreubo01    1998  151  497   68  155  29   6  17  74.0  ...    NL   
3     abreubo01    1999  152  546  118  183  35  11  20  93.0  ...    NL   
4     abreubo01    2000  154  576  103  182  42  10  25  79.0  ...    NL   
...         ...     ...  ...  ...  ...  ...  ..  ..  ..   ...  ...   ...   
4793  zobribe01    2011  156  588   99  158  46   6  20  91.0  ...    AL   
4794  zobribe01    2012  157  560   88  151  39   7  20  74.0  ...    AL   
4795  zobribe

In [2]:
y = df['next_OPS']
X = df.drop(['next_OPS', 'playerID','yearID','birthYear','nameLast','nameGiven','lgID'], axis = 1)

In [3]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=.2, random_state=123)

LinearModel = LinearRegression()
LinearModel.fit(X_train, y_train)
y_preds = LinearModel.predict(X_test)

mse = mean_squared_error(y_test, y_preds)
linear_RMSE = np.sqrt(mse)
linear_r2 = r2_score(y_test, y_preds)
print(f'RMSE: {linear_RMSE}, R2_Score: {linear_r2}')

RMSE: 0.07859791570495928, R2_Score: 0.43139082542023865


In [4]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

RidgeCVModel = RidgeCV(alphas=[0.01, 0.1, 1.0, 10.0, 100.0])
RidgeCVModel.fit(X_train_scaled, y_train)
y_preds = RidgeCVModel.predict(X_test_scaled)

mse = mean_squared_error(y_test, y_preds)
ridge_RMSE = np.sqrt(mse)
ridge_r2 = r2_score(y_test, y_preds)
print(f'RMSE: {ridge_RMSE}, R2_Score: {ridge_r2}')
print(RidgeCVModel.alpha_)

RMSE: 0.07858909234867127, R2_Score: 0.43151848172475715
100.0


In [5]:
LassoCVModel = LassoCV(alphas=[0.01, 0.1, 1.0, 10.0, 100.0])
LassoCVModel.fit(X_train_scaled, y_train)
y_preds = LassoCVModel.predict(X_test_scaled)

mse = mean_squared_error(y_test, y_preds)
lasso_RMSE = np.sqrt(mse)
lasso_r2 = r2_score(y_test, y_preds)
print(f'RMSE: {lasso_RMSE}, R2_Score: {lasso_r2}')
print(LassoCVModel.alpha_)
print(dict(zip(X.columns, LassoCVModel.coef_)))

RMSE: 0.08190296026997111, R2_Score: 0.38256534435915535
0.01
{'G': np.float64(-0.0), 'AB': np.float64(-0.0), 'R': np.float64(0.0), 'H': np.float64(-0.0), '2B': np.float64(0.0), '3B': np.float64(-0.0), 'HR': np.float64(0.009848373079933458), 'RBI': np.float64(0.0), 'SB': np.float64(-0.0), 'CS': np.float64(-0.0), 'BB': np.float64(0.00887816805005809), 'SO': np.float64(0.0), 'IBB': np.float64(0.0022687319148726385), 'HBP': np.float64(0.0), 'SH': np.float64(-0.003178767821858576), 'SF': np.float64(0.0), 'GIDP': np.float64(0.0), 'weight': np.float64(0.0), 'height': np.float64(6.647489324506293e-05), 'salary': np.float64(0.0), 'age': np.float64(-0.0), 'OBP': np.float64(0.0), 'SLG': np.float64(0.001301063741456017), 'OPS': np.float64(0.0365575567716228), 'bats_B': np.float64(-0.0), 'bats_L': np.float64(0.0), 'bats_R': np.float64(-0.0)}


In [6]:
os.makedirs('models', exist_ok=True)

param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [5, 10],
    'min_samples_split': [2, 5, 10]
}

if os.path.exists('models/rf_grid_search.pkl'):
    grid_search = joblib.load('models/rf_grid_search.pkl')
else:
    grid_search = GridSearchCV(
        estimator=RandomForestRegressor(),
        param_grid=param_grid,
        cv=5,
        scoring='neg_root_mean_squared_error'
    )
    grid_search.fit(X_train, y_train)
    joblib.dump(grid_search, 'models/rf_grid_search.pkl')

print(f"Best params: {grid_search.best_params_}")

y_preds = grid_search.best_estimator_.predict(X_test)

mse = mean_squared_error(y_test, y_preds)
forest_RMSE = np.sqrt(mse)
forest_r2 = r2_score(y_test, y_preds)
print(f'RMSE: {forest_RMSE}, R2 Score: {forest_r2}')

Best params: {'max_depth': 10, 'min_samples_split': 10, 'n_estimators': 200}
RMSE: 0.07933914102069828, R2 Score: 0.4206156068554491


In [7]:
os.makedirs('models', exist_ok=True)

param_grid = {
    'n_estimators': [100, 200],
    'learning_rate': [.01, .1],
    'max_depth': [3, 5],
    'subsample': [.8, 1.0]
}

if os.path.exists('models/xgb_grid_search.pkl'):
    xgb_grid_search = joblib.load('models/xgb_grid_search.pkl')
else:
    xgb_grid_search = GridSearchCV(
        estimator=XGBRegressor(),
        param_grid=param_grid,
        cv=5,
        scoring='neg_root_mean_squared_error'
    )
    xgb_grid_search.fit(X_train, y_train)
    joblib.dump(xgb_grid_search, 'models/xgb_grid_search.pkl')

print(f"Best params: {xgb_grid_search.best_params_}")

y_preds = xgb_grid_search.best_estimator_.predict(X_test)

mse = mean_squared_error(y_test, y_preds)
xgb_RMSE = np.sqrt(mse)
xgb_r2 = r2_score(y_test, y_preds)
print(f'RMSE: {xgb_RMSE}, R2 Score: {xgb_r2}')

Best params: {'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 100, 'subsample': 1.0}
RMSE: 0.07955875022820927, R2 Score: 0.4174037181924555


In [8]:
results = {'Model': ['Linear Regression', 'Ridge Regression','Lasso Regression', 'Random Forest', 'XG Boost'],
           'RMSE': [linear_RMSE, ridge_RMSE,lasso_RMSE, forest_RMSE, xgb_RMSE],
           'R2 Score': [linear_r2, ridge_r2, lasso_r2, forest_r2, xgb_r2]}
results_df = pd.DataFrame(results).sort_values('RMSE')

results_df

,Model,RMSE,R2 Score
1,Ridge Regression,0.078589,0.431518
0,Linear Regression,0.078598,0.431391
3,Random Forest,0.079339,0.420616
4,XG Boost,0.079559,0.417404
2,Lasso Regression,0.081903,0.382565


In [13]:
X_train_scaled_df = pd.DataFrame(X_train_scaled, columns=X_train.columns)
X_test_scaled_df = pd.DataFrame(X_test_scaled, columns=X_test.columns)

shap_explainer = shap.LinearExplainer(RidgeCVModel, X_train_scaled_df)
shap_values = shap_explainer.shap_values(X_test_scaled_df)
shap.summary_plot(shap_values, X_test_scaled_df, plot_type = 'bar', show=False)
plt.savefig('outputs/figures/shap_ops.png', dpi=150, bbox_inches='tight')
plt.close()

In [10]:
def get_next_ops_prediction(playerID):
    if playerID in df['playerID'].values:
        player_max_year = df[df['playerID'] == playerID]['yearID'].max()
        player_data = df[(df['playerID'] == playerID) & (df['yearID'] == player_max_year)]
        player_data.drop(['next_OPS', 'playerID','yearID','birthYear','nameLast','nameGiven','lgID'], axis = 1, inplace=True)
        player_data_scaled = scaler.transform(player_data)
        OPS_pred = RidgeCVModel.predict(player_data_scaled)
        return f'Predicted OPS: {OPS_pred}'
    else:
        return f'Sorry that playerID does not exist'
    
get_next_ops_prediction('troutmi01')
    

'Predicted OPS: [0.92329455]'

NameError: name 'conn' is not defined